# Fragrantica Parfum Chatbot
**Project:** Fragheadsunited — AI Fragrance Recommendation Engine  
**Doel:** RAG-pipeline met ChromaDB + chatbot interface  
**Input:** `fragrantica_final.csv` (output van de data pipeline notebook)

## Architectuur
```
Gebruiker typt beschrijving
        ↓
Sentence Transformer → embedding
        ↓
ChromaDB → top-K meest vergelijkbare parfums ophalen
        ↓
LLM (OpenAI / lokaal) → mooi antwoord formuleren
        ↓
Chatbot geeft aanbevelingen terug
```

---

## 0. Installatie

In [1]:
# Installeer benodigde packages (eenmalig)
!pip install chromadb sentence-transformers pandas openai gradio

^C



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
%pip install chromadb

import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

print('Imports geladen.')

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Imports geladen.


In [5]:
import chromadb


## 1. Data Inladen

> Zorg dat `fragrantica_final.csv` in dezelfde map staat (output van de pipeline notebook).

In [6]:
# ============================================================
# AANPASSEN: pad naar je dataset
PATH = 'fragrantica_final.csv'
# ============================================================

df = pd.read_csv(PATH)

# Verwijder rijen zonder embedding tekst
df = df.dropna(subset=['embedding_text']).reset_index(drop=True)

print(f'Dataset geladen: {len(df):,} parfums')
print(f'Kolommen: {df.columns.tolist()}')

Dataset geladen: 84,144 parfums
Kolommen: ['rating', 'notes', 'designer', 'reviews', 'description', 'url', 'title', 'brand_clean', 'gender', 'notes_list', 'reviews_list', 'notes_text', 'reviews_text', 'review_count', 'embedding_text', 'text_length']


---
## 2. Embedding Model Laden

In [7]:
# Lichtgewicht multilinguaal model snel en goed genoeg voor PoC
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model geladen.')

Embedding model geladen.


---
## 3. ChromaDB Vectordatabase Opbouwen

ChromaDB slaat de embeddings op zodat je snel kunt zoeken zonder alles opnieuw te berekenen.

>  De eerste keer duurt dit even (embeddings berekenen voor de hele dataset).  
> Daarna laden we de opgeslagen database meteen.

In [8]:
import os

# ChromaDB opslaan op schijf zodat je het niet elke keer opnieuw hoeft te berekenen
CHROMA_PATH = './chroma_db'
COLLECTION_NAME = 'fragrantica'

# Persistente client
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Controleer of collection al bestaat
existing = [c.name for c in chroma_client.list_collections()]

if COLLECTION_NAME in existing:
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print(f'Bestaande collection geladen: {collection.count():,} parfums')
else:
    print('Nieuwe collection aanmaken en embeddings berekenen...')
    print('Dit kan enkele minuten duren voor de volledige dataset.')
    
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={'hnsw:space': 'cosine'}   # cosine similarity
    )
    
    # In batches verwerken (ChromaDB heeft een limiet per batch)
    BATCH_SIZE = 500
    total = len(df)
    
    for start in range(0, total, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total)
        batch = df.iloc[start:end]
        
        # Embeddings berekenen
        texts = batch['embedding_text'].tolist()
        embeddings = model.encode(texts, batch_size=64, show_progress_bar=False).tolist()
        
        # Metadata voor elk parfum
        metadatas = []
        for _, row in batch.iterrows():
            metadatas.append({
                'title':      str(row.get('title', '')),
                'brand':      str(row.get('brand_clean', '')),
                'gender':     str(row.get('gender', '')),
                'rating':     float(row['rating']) if pd.notna(row.get('rating')) else 0.0,
                'notes':      str(row.get('notes_text', '')),
                'url':        str(row.get('url', '')),
            })
        
        # IDs moeten uniek en string zijn
        ids = [str(i) for i in batch.index.tolist()]
        
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        
        if (start // BATCH_SIZE) % 10 == 0:
            print(f'  Verwerkt: {end:,} / {total:,}')
    
    print(f'\nKlaar! {collection.count():,} parfums opgeslagen in ChromaDB.')

Bestaande collection geladen: 84,144 parfums


---
## 4. Zoekfunctie

De kern van de RAG-pipeline: gegeven een vrije tekstquery, haal de meest relevante parfums op.

In [9]:
def zoek_parfums(query: str, top_k: int = 5, gender_filter: str = None) -> list:
    """
    Zoek de meest relevante parfums op basis van een tekstquery.
    
    Parameters:
        query         : Vrije tekstbeschrijving (bijv. 'fris citrus zomers')
        top_k         : Aantal resultaten
        gender_filter : Optioneel filteren op 'men', 'women', 'unisex'
    
    Returns:
        Lijst van dicts met parfuminformatie
    """
    # Query embedden
    query_embedding = model.encode([query]).tolist()
    
    # Optioneel gender filter als ChromaDB where-clause
    where = None
    if gender_filter and gender_filter in ('men', 'women', 'unisex'):
        where = {'gender': {'$eq': gender_filter}}
    
    # Zoeken in ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where,
        include=['metadatas', 'distances']
    )
    
    # Resultaten opmaken
    parfums = []
    for meta, dist in zip(results['metadatas'][0], results['distances'][0]):
        parfums.append({
            'title':      meta.get('title', ''),
            'brand':      meta.get('brand', ''),
            'gender':     meta.get('gender', ''),
            'rating':     meta.get('rating', 0),
            'notes':      meta.get('notes', ''),
            'url':        meta.get('url', ''),
            'similarity': round(1 - dist, 3),   # ChromaDB geeft afstand terug, wij willen similariteit
        })
    
    return parfums


# Snelle test
resultaten = zoek_parfums('fris citrus zomers strand', top_k=3)
for r in resultaten:
    print(f"  {r['title']} | {r['notes'][:60]}... | sim={r['similarity']}")

  Fretnot Origins for women and men | Tangerine, Mandarin Orange, Orange, Lemon, Lime... | sim=0.5
  Citrus Amber Being Frenshe for women | Clementine, Amber, Lemon, Orange Blossom... | sim=0.476
  Cannabis Fruttata Bois 1920 for women and men | Rosemary, Oregano, Fig Leaf, cannabis, Blueberry, Lily-of-th... | sim=0.471


---
## 5. LLM-antwoord Genereren

We sturen de gevonden parfums als context naar een LLM, die er een vriendelijk chatbot-antwoord van maakt.

### Optie A: OpenAI API (beste kwaliteit)
### Optie B: Simpele template zonder LLM (geen API key nodig)

In [10]:
# ============================================================
# KIES JE OPTIE:
# 'openai'   → OpenAI API (vul OPENAI_API_KEY in)
# 'template' → Geen LLM, simpele opmaak (gratis)
LLM_MODE = 'template'
#OPENAI_API_KEY = 'sk-...'   # alleen nodig bij 'openai'
# ============================================================


def genereer_antwoord(query: str, parfums: list) -> str:
    """
    Genereer een chatbot-antwoord op basis van de gevonden parfums.
    """
    if not parfums:
        return 'Sorry, ik kon geen passende parfums vinden voor jouw beschrijving. Probeer andere zoekwoorden.'
    
    if LLM_MODE == 'openai':
        return _antwoord_openai(query, parfums)
    else:
        return _antwoord_template(query, parfums)


def _antwoord_template(query: str, parfums: list) -> str:
    regels = [f'Based on your description **"{query}"** I recommend the following perfumes:\n']

    for i, p in enumerate(parfums, 1):
        rating_str = f"{p['rating']:.2f} / 5.00" if p['rating'] else ''
        rating_regel = f"Rating: {rating_str}  |  " if rating_str else ''
        notes_str = p['notes'][:80] + '...' if len(p['notes']) > 80 else p['notes']
        uitleg = _genereer_uitleg(query, p)
        review = _haal_review_op(p['title'])

        regels.append(
            f"**{i}. {p['title']}**\n"
            f"{rating_regel}Gender: {p['gender']}  |  Match: {p['similarity']:.0%}\n"
            f"Notes: {notes_str}\n"
            f"Why this one: {uitleg}\n"
            f"Review: *\"{review}\"*\n"
        )

    return '\n'.join(regels)


def _haal_review_op(title: str) -> str:
    """Haal een echte review op uit de dataset."""
    rij = df[df['title'] == title]
    if rij.empty:
        return 'No reviews available.'

    reviews_text = rij.iloc[0].get('reviews_text', '')
    if not reviews_text or str(reviews_text) == 'nan':
        return 'No reviews available.'

    # Eerste zin van de review pakken
    eerste_zin = str(reviews_text).split('.')[0].strip()
    if len(eerste_zin) < 20:
        eerste_zin = str(reviews_text)[:200]

    return eerste_zin[:200]


def _genereer_uitleg(query: str, parfum: dict) -> str:
    query_lower = query.lower()
    notes_lower = parfum['notes'].lower()
    redenen = []

    if any(w in query_lower for w in ['fresh', 'fris', 'citrus', 'summer', 'zomer', 'beach', 'strand', 'light']):
        matches = [n for n in ['citrus', 'lemon', 'bergamot', 'lime', 'orange', 'grapefruit', 'neroli', 'yuzu']
                   if n in notes_lower]
        if matches:
            redenen.append(f"contains fresh citrus notes ({', '.join(matches[:2])})")

    if any(w in query_lower for w in ['wood', 'hout', 'woody', 'oud', 'dark', 'donker', 'deep']):
        matches = [n for n in ['oud', 'cedar', 'sandalwood', 'wood', 'patchouli', 'vetiver']
                   if n in notes_lower]
        if matches:
            redenen.append(f"has deep woody notes ({', '.join(matches[:2])})")

    if any(w in query_lower for w in ['sweet', 'zoet', 'vanilla', 'vanille', 'warm', 'gourmand']):
        matches = [n for n in ['vanilla', 'caramel', 'honey', 'amber', 'tonka', 'musk']
                   if n in notes_lower]
        if matches:
            redenen.append(f"carries warm sweet notes ({', '.join(matches[:2])})")

    if any(w in query_lower for w in ['floral', 'bloem', 'rose', 'roos', 'jasmine', 'jasmijn', 'feminine', 'vrouwelijk']):
        matches = [n for n in ['rose', 'jasmine', 'lily', 'peony', 'iris', 'violet', 'orchid']
                   if n in notes_lower]
        if matches:
            redenen.append(f"is rich in floral notes ({', '.join(matches[:2])})")

    if any(w in query_lower for w in ['spicy', 'kruidig', 'pittig', 'oriental', 'oriëntaals']):
        matches = [n for n in ['pepper', 'saffron', 'cinnamon', 'cardamom', 'ginger', 'clove']
                   if n in notes_lower]
        if matches:
            redenen.append(f"has spicy notes ({', '.join(matches[:2])})")

    if not redenen:
        similarity_pct = int(parfum['similarity'] * 100)
        redenen.append(f"closely matches your description (semantic match: {similarity_pct}%)")

    return 'This perfume fits because it ' + ' and '.join(redenen) + '.'

---
## 6. Chatbot Functie

Alles samenvoegen tot één functie.

In [11]:
def chatbot(query: str, top_k: int = 5, gender_filter: str = None) -> str:
    """
    Parfum aanbevelings-chatbot.
    
    Parameters:
        query         : Wat zoekt de gebruiker?
        top_k         : Hoeveel parfums ophalen?
        gender_filter : 'men', 'women', 'unisex' of None
    """
    print(f' Zoeken naar: "{query}"...')
    
    # Stap 1: Relevante parfums ophalen uit ChromaDB
    parfums = zoek_parfums(query, top_k=top_k, gender_filter=gender_filter)
    
    # Stap 2: LLM-antwoord genereren
    antwoord = genereer_antwoord(query, parfums)
    
    return antwoord


# Test 1
print('=' * 60)
antwoord = chatbot('fris citrus voor de zomer aan het strand')
print(antwoord)

 Zoeken naar: "fris citrus voor de zomer aan het strand"...
Based on your description **"fris citrus voor de zomer aan het strand"** I recommend the following perfumes:

**1. Eau de Cartier Zeste de Soleil Cartier for women and men**
Gender: unisex  |  Match: 45%
Notes: Yuzu, Passionfruit, Mint
Why this one: This perfume fits because it contains fresh citrus notes (yuzu).
Review: *"Really great yuzu scent, with added tanginess from the passionfruit and accented briefly by mint"*

**2. Feuilles & Fleurs d'Oranger Galimard for women and men**
Gender: unisex  |  Match: 45%
Notes: Orange Leaf, Petitgrain, Orange Blossom, Neroli
Why this one: This perfume fits because it contains fresh citrus notes (orange, neroli).
Review: *"This is one beautiful juice"*

**3. Fleur d'Oranger Evody Parfums for women and men**
Gender: unisex  |  Match: 45%
Notes: Bergamot, Mandarin Orange, Neroli, Orange Blossom, Petitgrain, Musk, Ambroxan, C...
Why this one: This perfume fits because it contains fresh citr

In [12]:
# Test 2
print('=' * 60)
antwoord = chatbot('donker houtachtig oud oosters', gender_filter='men')
print(antwoord)

 Zoeken naar: "donker houtachtig oud oosters"...
Based on your description **"donker houtachtig oud oosters"** I recommend the following perfumes:

**1. Öslo Viking for men**
Gender: men  |  Match: 32%
Notes: Musk, Sea Notes, Oakmoss, Cardamom
Why this one: This perfume fits because it closely matches your description (semantic match: 32%).
Review: *"A perfect example of why you should never, ever, use additional letters of an alphabet in a language you're not familiar with and to use a proofreader before launching a new scent"*

**2. Boss Bottled Oud Aromatic Hugo Boss for men**
Gender: men  |  Match: 31%
Notes: Orange Blossom, Myrhh, Agarwood (Oud)
Why this one: This perfume fits because it has deep woody notes (oud, wood).
Review: *"smells nice and everything but, who the heck decided to call it "Oud Aromatic" and where is the aromatic note in this coming from? I was in a perfume shop doing some sampling"*

**3. Leather Oud Attar The Dua Brand for men**
Gender: men  |  Match: 31%
No

In [13]:

# Test 3
print('=' * 60)
antwoord = chatbot('zoet bloemig roze vrouwelijk', gender_filter='women')
print(antwoord)

 Zoeken naar: "zoet bloemig roze vrouwelijk"...
Based on your description **"zoet bloemig roze vrouwelijk"** I recommend the following perfumes:

**1. Kredo (Кредо) Dzintars for women**
Gender: women  |  Match: 34%
Notes: Bergamot, Lily-of-the-Valley, Rose, Violet, Ylang-Ylang, Carnation, Jasmine, Tub...
Why this one: This perfume fits because it is rich in floral notes (rose, jasmine).
Review: *"No reviews available."*

**2. Fleur de Thé Karl Lagerfeld for women**
Gender: women  |  Match: 33%
Notes: Citruses, Jasmine Sambac, Camellia
Why this one: This perfume fits because it is rich in floral notes (jasmine).
Review: *"Similar to, but much better than Elizabeth Arden's Green Tea and l'Eau par Kenzo pour Homme"*

**3. Latvija (Latvia) Dzintars for women**
Gender: women  |  Match: 32%
Notes: Lily-of-the-Valley, Lime (Linden Blossom), Violet, Cyclamen, Jasmine, Vetiver, Y...
Why this one: This perfume fits because it carries warm sweet notes (amber, musk) and is rich in floral notes (ro

---
## 7. Evaluatie

Drie evaluatiemethodes om de kwaliteit van de RAG-pipeline te meten:
- **Optie 1:** Similarity score analyse (automatisch)
- **Optie 2:** Precision@K (formeel metriek)

In [17]:
# === OPTIE 1: Similarity Score Analyse ===
test_cases = [
    {"query": "fresh citrus summer beach", "expected_notes": ["citrus", "lemon", "bergamot", "lime", "orange"]},
    {"query": "dark woody oud oriental",   "expected_notes": ["oud", "cedar", "sandalwood", "patchouli"]},
    {"query": "sweet vanilla warm gourmand", "expected_notes": ["vanilla", "caramel", "amber", "tonka"]},
    {"query": "floral rose jasmine feminine", "expected_notes": ["rose", "jasmine", "lily", "iris", "peony"]},
    {"query": "spicy pepper saffron",      "expected_notes": ["pepper", "saffron", "cinnamon", "cardamom"]},
]

print("=== Similarity Score Analyse ===\n")
for case in test_cases:
    results = zoek_parfums(case["query"], top_k=5)
    hits = sum(
        any(note in r["notes"].lower() for note in case["expected_notes"])
        for r in results
    )
    avg_sim = sum(r["similarity"] for r in results) / len(results)
    print(f'Query: "{case["query"]}"')
    print(f'  Relevante resultaten: {hits}/5 | Gem. similarity: {avg_sim:.3f}\n')

=== Similarity Score Analyse ===

Query: "fresh citrus summer beach"
  Relevante resultaten: 5/5 | Gem. similarity: 0.582

Query: "dark woody oud oriental"
  Relevante resultaten: 4/5 | Gem. similarity: 0.555

Query: "sweet vanilla warm gourmand"
  Relevante resultaten: 4/5 | Gem. similarity: 0.551

Query: "floral rose jasmine feminine"
  Relevante resultaten: 5/5 | Gem. similarity: 0.701

Query: "spicy pepper saffron"
  Relevante resultaten: 5/5 | Gem. similarity: 0.599



In [19]:
# === OPTIE 2: PrecisionK ===
def precision_at_k(query, expected_notes, k=5):
    results = zoek_parfums(query, top_k=k)
    relevant = sum(
        any(n in r["notes"].lower() for n in expected_notes)
        for r in results
    )
    return relevant / k

print("=== PrecisionK Evaluatie ===\n")
pk_results = []
for case in test_cases:
    score = precision_at_k(case["query"], case["expected_notes"], k=5)
    pk_results.append({"query": case["query"], "precision5": score})
    print(f'"{case["query"]}": P@5 = {score:.2f}')

pk_df = pd.DataFrame(pk_results)
print(f'\nGemiddelde Precision5: {pk_df["precision5"].mean():.2f}')

=== PrecisionK Evaluatie ===

"fresh citrus summer beach": P@5 = 1.00
"dark woody oud oriental": P@5 = 0.80
"sweet vanilla warm gourmand": P@5 = 0.80
"floral rose jasmine feminine": P@5 = 1.00
"spicy pepper saffron": P@5 = 1.00

Gemiddelde Precision5: 0.92


---
## 7.  Chatbot Interface

Een eenvoudige webinterface voor de chatbot.  
Run deze cel en open de link die verschijnt.

In [14]:
%pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Simpele interactieve chatbot zonder Gradio
from ipywidgets import widgets
from IPython.display import display, Markdown, clear_output

# UI elementen
tekst_input  = widgets.Text(placeholder='Beschrijf het parfum dat je zoekt...', layout=widgets.Layout(width='60%'))
gender_dropdown = widgets.Dropdown(options=['Alle', 'women', 'men', 'unisex'], value='Alle', description='Gender:')
zoek_knop    = widgets.Button(description='Zoeken 🔍', button_style='primary')
output       = widgets.Output()

def on_zoeken(b):
    with output:
        clear_output()
        query  = tekst_input.value.strip()
        gender = None if gender_dropdown.value == 'Alle' else gender_dropdown.value
        if not query:
            print('Vul eerst een beschrijving in.')
            return
        antwoord = chatbot(query, top_k=5, gender_filter=gender)
        display(Markdown(antwoord))

zoek_knop.on_click(on_zoeken)

display(widgets.HBox([tekst_input, gender_dropdown, zoek_knop]))
display(output)

Output()